# Week 3 — PySpark for Attendance Analysis
### Employee Attendance & Productivity Tracker Capstone

In [1]:
from pyspark.sql import SparkSession

spark = SparkSession.builder \
    .appName('AttendanceWeek3') \
    .master('local[*]') \
    .getOrCreate()

print('Spark version:', spark.version)


Spark version: 4.1.2


## Load cleaned attendance data (from Week 2 output)

In [2]:
df = spark.read.csv('attendance_cleaned.csv', header=True, inferSchema=True)
df.printSchema()
df.show(10)


root
 |-- employee_id: integer (nullable = true)
 |-- employee_name: string (nullable = true)
 |-- department: string (nullable = true)
 |-- work_date: date (nullable = true)
 |-- clock_in: timestamp (nullable = true)
 |-- clock_out: timestamp (nullable = true)
 |-- tasks_completed: integer (nullable = true)
 |-- tasks_assigned: integer (nullable = true)
 |-- status: string (nullable = true)
 |-- work_hours: double (nullable = true)
 |-- break_time_hrs: double (nullable = true)
 |-- late_login: boolean (nullable = true)
 |-- productivity_score: double (nullable = true)

+-----------+-------------+-----------+----------+-------------------+-------------------+---------------+--------------+-------+----------+--------------+----------+------------------+
|employee_id|employee_name| department| work_date|           clock_in|          clock_out|tasks_completed|tasks_assigned| status|work_hours|break_time_hrs|late_login|productivity_score|
+-----------+-------------+-----------+----------+-

## Filter: Late Logins
Using the `late_login` flag already computed in Week 2.

In [3]:
from pyspark.sql.functions import col

late_logins = df.filter(col('late_login') == True)
late_logins.select('employee_id', 'employee_name', 'department', 'work_date', 'clock_in', 'late_login').show()

print('Total late login instances:', late_logins.count())


+-----------+-------------+-----------+----------+-------------------+----------+
|employee_id|employee_name| department| work_date|           clock_in|late_login|
+-----------+-------------+-----------+----------+-------------------+----------+
|          3|   Amit Kumar|      Sales|2026-06-01|1900-01-01 09:30:00|      true|
|          7|  Arjun Verma|Engineering|2026-06-01|1900-01-01 09:50:00|      true|
|          5|   Farhan Ali|      Sales|2026-06-02|1900-01-01 10:00:00|      true|
|          8|   Meera Nair|  Marketing|2026-06-02|1900-01-01 09:20:00|      true|
|          1| Rahul Sharma|Engineering|2026-06-03|1900-01-01 10:15:00|      true|
|          3|   Amit Kumar|      Sales|2026-06-03|1900-01-01 09:45:00|      true|
|          6|   Neha Singh|  Marketing|2026-06-03|1900-01-01 10:30:00|      true|
|          3|   Amit Kumar|      Sales|2026-06-04|1900-01-01 10:20:00|      true|
|          5|   Farhan Ali|      Sales|2026-06-04|1900-01-01 09:30:00|      true|
|          3|   

## Filter: Absences

In [4]:
absences = df.filter(col('status') == 'Absent')
absences.select('employee_id', 'employee_name', 'department', 'work_date', 'status').show()

print('Total absence records:', absences.count())


+-----------+-------------+-----------+----------+------+
|employee_id|employee_name| department| work_date|status|
+-----------+-------------+-----------+----------+------+
|          5|   Farhan Ali|      Sales|2026-06-01|Absent|
|          3|   Amit Kumar|      Sales|2026-06-02|Absent|
|          5|   Farhan Ali|      Sales|2026-06-03|Absent|
|          7|  Arjun Verma|Engineering|2026-06-04|Absent|
|          5|   Farhan Ali|      Sales|2026-06-05|Absent|
+-----------+-------------+-----------+----------+------+

Total absence records: 5


## Group by Department — Average Work Hours and Productivity

In [5]:
from pyspark.sql.functions import avg, round as spark_round, sum as spark_sum, count, when

dept_summary = df.groupBy('department').agg(
    spark_round(avg('work_hours'), 2).alias('avg_work_hours'),
    spark_round(avg('productivity_score'), 2).alias('avg_productivity_score'),
    spark_sum(when(col('status') == 'Absent', 1).otherwise(0)).alias('total_absences'),
    spark_sum(when(col('late_login') == True, 1).otherwise(0)).alias('total_late_logins'),
    count('employee_id').alias('total_records')
).orderBy(col('avg_productivity_score').desc())

dept_summary.show()


+-----------+--------------+----------------------+--------------+-----------------+-------------+
| department|avg_work_hours|avg_productivity_score|total_absences|total_late_logins|total_records|
+-----------+--------------+----------------------+--------------+-----------------+-------------+
|Engineering|          8.89|                  0.65|             1|                2|           15|
|  Marketing|          8.33|                  0.55|             0|                2|           10|
|         HR|           9.0|                  0.38|             0|                0|            5|
|      Sales|          7.95|                  0.16|             4|                6|           10|
+-----------+--------------+----------------------+--------------+-----------------+-------------+



## Identify Attendance Issues by Department
Departments with above-average absences or late logins flagged as needing attention.

In [6]:
avg_absences = dept_summary.agg(avg('total_absences')).collect()[0][0]
avg_late = dept_summary.agg(avg('total_late_logins')).collect()[0][0]

print('Avg absences across departments:', round(avg_absences, 2))
print('Avg late logins across departments:', round(avg_late, 2))

attendance_issues = dept_summary.filter(
    (col('total_absences') > avg_absences) | (col('total_late_logins') > avg_late)
).orderBy(col('total_absences').desc())

attendance_issues.show()


Avg absences across departments: 1.25
Avg late logins across departments: 2.5
+----------+--------------+----------------------+--------------+-----------------+-------------+
|department|avg_work_hours|avg_productivity_score|total_absences|total_late_logins|total_records|
+----------+--------------+----------------------+--------------+-----------------+-------------+
|     Sales|          7.95|                  0.16|             4|                6|           10|
+----------+--------------+----------------------+--------------+-----------------+-------------+



## Employee-Level Attendance Issue Report
Employees with 2+ absences or 2+ late logins, flagged for HR review.

In [7]:
employee_summary = df.groupBy('employee_id', 'employee_name', 'department').agg(
    spark_sum(when(col('status') == 'Absent', 1).otherwise(0)).alias('total_absences'),
    spark_sum(when(col('late_login') == True, 1).otherwise(0)).alias('total_late_logins'),
    spark_round(avg('productivity_score'), 2).alias('avg_productivity_score')
)

flagged_employees = employee_summary.filter(
    (col('total_absences') >= 2) | (col('total_late_logins') >= 2)
).orderBy(col('total_absences').desc())

flagged_employees.show()


+-----------+-------------+----------+--------------+-----------------+----------------------+
|employee_id|employee_name|department|total_absences|total_late_logins|avg_productivity_score|
+-----------+-------------+----------+--------------+-----------------+----------------------+
|          5|   Farhan Ali|     Sales|             3|                2|                  0.12|
|          3|   Amit Kumar|     Sales|             1|                4|                  0.21|
+-----------+-------------+----------+--------------+-----------------+----------------------+



## Save Output — Attendance Issues by Department

In [8]:
dept_summary.toPandas().to_csv('output_department_attendance_summary.csv', index=False)
flagged_employees.toPandas().to_csv('output_flagged_employees.csv', index=False)

print('Saved department_attendance_summary and flagged_employees as CSV')


Saved department_attendance_summary and flagged_employees as CSV


In [9]:
spark.stop()
